# Chapter 10: Contexts, Expressions, Outputs & `needs` (Reference)

## Learning Objectives

- Resolve dotted github.* context paths against a representative context
- Evaluate the four if: status functions against a simulated job status
- Simulate a job output crossing the needs.<job>.outputs boundary
- Explain why github.sha differs from github.event.pull_request.head.sha

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")


## 1. Resolving the `github` Context

The next cell resolves four dotted paths against `SAMPLE_GITHUB_CONTEXT`. You should see `github.sha` (the synthetic merge commit) printed alongside `github.event.pull_request.head.sha` (the author's real commit) -- note they differ.

In [ ]:
from labs.lab_10_job_outputs import SAMPLE_GITHUB_CONTEXT, resolve_context_path

for path in ["sha", "event.pull_request.head.sha", "event.pull_request.base.ref", "actor"]:
    value = resolve_context_path(SAMPLE_GITHUB_CONTEXT, path)
    print(f"github.{path:<32} = {value}")


## 2. The Four Status Functions

The next cell evaluates `success()`, `failure()`, `cancelled()`, `always()` against a simulated `job_status='failure'`. You should see only `failure()` and `always()` return `True`.

In [ ]:
from labs.lab_10_job_outputs import evaluate_status_function

for fn in ["success", "failure", "cancelled", "always"]:
    runs = evaluate_status_function(fn, job_status="failure")
    print(f"if: {fn}()  -> runs = {runs}")


## 3. `needs`/Outputs: Passing Gate 3's Score Downstream

The next cell simulates Gate 3 publishing a `risk_score` output and a downstream job reading it via `needs.gate3.outputs.risk_score`. You should see the output arrive as a string, and `downstream_would_run = True` since 66.0 <= 70.

In [ ]:
from labs.lab_10_job_outputs import simulate_needs_output_passing

result = simulate_needs_output_passing(risk_score=66.0, threshold=70.0)
print(f"gate3 outputs: {result['gate3_outputs']}")
print(f"downstream job would run: {result['downstream_would_run']}")


## Takeaways & Next Steps

This notebook's takeaway is the contrast in Section 1 -- `github.sha` vs `github.event.pull_request.head.sha` -- re-read it before moving on.

In [ ]:
print("See Chapter 10 Section 9 for why needs.*.outputs.* values are always strings.")


---

📖 **Reading companion:** [Chapter 10: Contexts, Expressions, Outputs & `needs`](../learning_modules/chapter_10_contexts_expressions.md)
